# Value Investing

Value investing ranks companies by how cheap they appear relative to fundamentals such as earnings, book value, or EBITDA.

Abbreviations used in this notebook:

- **P/E**: Price to Earnings.
- **EV**: Enterprise Value, the value of the operating business before subtracting net debt.
- **P/B**: Price to Book.
- **EV/EBITDA**: Enterprise Value divided by Earnings Before Interest, Taxes, Depreciation, and Amortization.
- **EBITDA**: Earnings Before Interest, Taxes, Depreciation, and Amortization.
- **IR**: Information Ratio.

## 1. Intuition

A value strategy looks for companies where the market price is low relative to fundamentals. The core belief is that cheap assets can outperform if the market eventually recognizes their value.

The danger is the value trap: a stock can be cheap because the business is deteriorating.

## 2. Mathematics

**Value composite score:**

$$
Score = z(-P/E) + z(-P/B) + z(-EV/EBITDA)
$$

Where:

- $EV$ = enterprise value, the value of the operating business
- $P/E$ = price-to-earnings multiple
- $P/B$ = price-to-book multiple
- $EBITDA$ = earnings before interest, taxes, depreciation, and amortization
- $EV/EBITDA$ = enterprise value to EBITDA multiple
- $Score$ = composite ranking score
- $z(\cdot)$ = standardized z-score transformation

**Equal-weight portfolio return:**

$$
R_{p,t} = \frac{1}{N}\sum_i R_{i,t}
$$

Where:

- Variables are defined in the surrounding text and implementation below

**Active return:**

$$
R_{active,t} = R_{strategy,t} - R_{benchmark,t}
$$

Where:

- Variables are defined in the surrounding text and implementation below

## 3. Implementation

We rank a synthetic stock universe by valuation multiples and build a portfolio from the cheapest quartile.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "05_strategies" / "strategy_utils.py"
spec = importlib.util.spec_from_file_location("strategy_utils", helper_path)
strategy_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(strategy_utils)

plt.style.use("seaborn-v0_8-whitegrid")
prices, returns, fundamentals = strategy_utils.generate_strategy_universe()
benchmark = returns.mean(axis=1)

value_data = fundamentals.copy()
value_data["value_score"] = (
    strategy_utils.zscore(value_data["pe"], higher_is_better=False)
    + strategy_utils.zscore(value_data["price_book"], higher_is_better=False)
    + strategy_utils.zscore(value_data["ev_ebitda"], higher_is_better=False)
)
value_data = value_data.sort_values("value_score", ascending=False)
selected = value_data.head(6)["ticker"].tolist()
value_returns = strategy_utils.equal_weight_return(returns, selected)

value_data.head(10)

In [ ]:
summary = pd.DataFrame({
    "value_strategy": strategy_utils.performance_summary(value_returns, benchmark),
    "benchmark": strategy_utils.performance_summary(benchmark),
})
summary.round(4)

## 4. Visualization

A value strategy should be inspected through both valuation scores and performance relative to the universe benchmark.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
value_data.head(10).plot(x="ticker", y="value_score", kind="bar", ax=axes[0], color="#2f6f8f", legend=False)
axes[0].set_title("Top Value Scores")
axes[0].set_xlabel("")
axes[0].tick_params(axis="x", rotation=35)

wealth = (1 + pd.DataFrame({"Value": value_returns, "Benchmark": benchmark})).cumprod()
wealth.plot(ax=axes[1], color=["#2f6f8f", "#9a6b2f"])
axes[1].set_title("Value Strategy vs Benchmark")
axes[1].set_ylabel("Growth of 1")
plt.tight_layout(); plt.show()

## 5. Application

In real research, value screens should be paired with quality checks. Cheap companies with weak balance sheets, declining margins, or poor capital returns may stay cheap for good reasons.

In [ ]:
selected_profile = value_data[value_data["ticker"].isin(selected)].set_index("ticker")[["pe", "price_book", "ev_ebitda", "roic", "debt_to_ebitda"]]
selected_profile.round(2)

## 6. Reflection

- Cheap is not the same as undervalued.
- Composite scores reduce reliance on one noisy metric.
- Value strategies can underperform for long periods.
- Quality filters can help avoid value traps.

Questions to answer after running the notebook:

1. Which stocks are selected by the value screen?
2. Do they also look financially healthy?
3. Did value outperform the benchmark?
4. What additional quality filter would you add?